# GOES Storm Watch — Export ONNX & Prepare Examples

Этот ноутбук делает две вещи:
1. **Экспорт модели** PyTorch → ONNX для браузерного инференса
2. **Подготовка примеров** — скачивает кропы штормовых регионов из GOES-16 S3

После выполнения скачай папки `model/` и `examples/` и положи в репозиторий.

## Часть 1: Экспорт модели в ONNX

### 1.1 Установка зависимостей

In [ ]:
!pip install -q onnx

### 1.2 Загрузи файл весов

Загрузи `best_model.pth` через панель файлов слева (иконка папки → Upload).
Или выполни ячейку ниже, чтобы загрузить через диалог.

In [ ]:
from google.colab import files
import os

if not os.path.exists("best_model.pth"):
    print("Загрузи best_model.pth:")
    uploaded = files.upload()
else:
    print("best_model.pth уже на месте")

### 1.3 Экспорт в ONNX

In [ ]:
import json
import os
import torch
import torch.nn as nn
from torchvision import models
import onnx

# ── Нормализация (из обучения) ──
NORM_STATS = {
    "channels": ["CMI_C07", "CMI_C09", "CMI_C13", "CMI_C14", "CMI_C15"],
    "means": [288.66, 245.55, 278.85, 278.19, 275.79],
    "stds":  [19.17,  12.98,  23.49,  23.76,  23.10],
    "patch_size": 64,
    "stride": 48,
    "convective_threshold": 0.5
}

def build_model():
    model = models.resnet18(weights=None)
    old_conv = model.conv1
    model.conv1 = nn.Conv2d(5, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        model.conv1.weight[:, :3] = old_conv.weight
        model.conv1.weight[:, 3] = old_conv.weight.mean(dim=1)
        model.conv1.weight[:, 4] = old_conv.weight.mean(dim=1)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

# Загрузка весов
model = build_model()
state = torch.load("best_model.pth", map_location="cpu")
model.load_state_dict(state)
model.eval()
print("Веса загружены")

# Экспорт
os.makedirs("model", exist_ok=True)
dummy = torch.randn(1, 5, 64, 64)

torch.onnx.export(
    model, dummy, "model/resnet18_goes.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=13,
    dynamo=False
)

# Сохранить статистику нормализации
with open("model/norm_stats.json", "w") as f:
    json.dump(NORM_STATS, f, indent=2)

# Проверка
onnx_model = onnx.load("model/resnet18_goes.onnx")
onnx.checker.check_model(onnx_model)

size_mb = os.path.getsize("model/resnet18_goes.onnx") / 1024 / 1024
print(f"Готово! Размер модели: {size_mb:.1f} MB")

### 1.4 Скачать модель

In [ ]:
from google.colab import files

files.download("model/resnet18_goes.onnx")
files.download("model/norm_stats.json")

---
## Часть 2: Подготовка примеров

Скачивает кропы штормовых регионов из публичного S3 GOES-16.

### 2.1 Установка зависимостей

In [ ]:
!pip install -q s3fs xarray h5netcdf

### 2.2 Скачать и нарезать кропы

In [ ]:
import os
import json
import numpy as np
import s3fs
import xarray as xr
from datetime import datetime

INPUT_CHANNELS = ['CMI_C07', 'CMI_C09', 'CMI_C13', 'CMI_C14', 'CMI_C15']
OUTPUT_DIR = "examples"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Интересные штормовые регионы
EXAMPLES = [
    {"date": "2022-06-12", "hour": "21", "name": "great_plains_storm",
     "title": "Great Plains Thunderstorms", "row": 200, "col": 800, "size": 256},
    {"date": "2022-07-05", "hour": "22", "name": "midwest_convection",
     "title": "Midwest Convective System", "row": 150, "col": 600, "size": 256},
    {"date": "2022-08-15", "hour": "20", "name": "gulf_storms",
     "title": "Gulf Coast Storms", "row": 400, "col": 500, "size": 256},
    {"date": "2019-05-20", "hour": "23", "name": "tornado_alley",
     "title": "Tornado Alley Supercells", "row": 250, "col": 700, "size": 256},
    {"date": "2021-07-28", "hour": "21", "name": "southeast_cells",
     "title": "Southeast Isolated Cells", "row": 350, "col": 400, "size": 256},
]

fs = s3fs.S3FileSystem(anon=True)

def find_snapshot(date, hour):
    year, month, day = date.split("-")
    doy = datetime(int(year), int(month), int(day)).timetuple().tm_yday
    mcmip_prefix = f"noaa-goes16/ABI-L2-MCMIPC/{year}/{doy:03d}/{hour}/"
    actpc_prefix = f"noaa-goes16/ABI-L2-ACTPC/{year}/{doy:03d}/{hour}/"
    mcmip_files = sorted(fs.ls(mcmip_prefix))
    actpc_files = sorted(fs.ls(actpc_prefix))
    if not mcmip_files or not actpc_files:
        return None, None
    return mcmip_files[0], actpc_files[0]

def open_nc(s3_path, variables=None):
    with fs.open(s3_path, 'rb') as f:
        ds = xr.open_dataset(f, engine='h5netcdf')
        if variables:
            ds = ds[variables]
        ds.load()
    return ds

manifest = []

for ex in EXAMPLES:
    print(f"\n=== {ex['title']} ({ex['date']} {ex['hour']}Z) ===")

    mcmip_path, actpc_path = find_snapshot(ex["date"], ex["hour"])
    if mcmip_path is None:
        print("  Файлы не найдены, пропускаем")
        continue

    print("  Загрузка MCMIPC...")
    ds_mcmip = open_nc(mcmip_path, INPUT_CHANNELS)
    print("  Загрузка ACTPC...")
    ds_actpc = open_nc(actpc_path, ['Phase'])

    r, c, s = ex["row"], ex["col"], ex["size"]

    # 5 каналов
    channels = []
    for ch in INPUT_CHANNELS:
        data = ds_mcmip[ch].values[r:r+s, c:c+s].astype(np.float32)
        if np.isnan(data).any():
            data[np.isnan(data)] = np.nanmean(data)
        channels.append(data)

    crop = np.stack(channels, axis=0)  # (5, H, W)

    # Ground truth маска
    phase = ds_actpc['Phase'].values[r:r+s, c:c+s]
    ctt = ds_mcmip['CMI_C13'].values[r:r+s, c:c+s]
    gt_mask = ((phase == 4) & (ctt < 220)).astype(np.float32)
    gt_mask[np.isnan(phase) | np.isnan(ctt)] = 0

    conv_pct = gt_mask.mean() * 100
    print(f"  Crop: {crop.shape}, convective: {conv_pct:.1f}%")

    npy_path = os.path.join(OUTPUT_DIR, f"{ex['name']}.npy")
    np.save(npy_path, crop)
    size_kb = os.path.getsize(npy_path) / 1024
    print(f"  Сохранено: {npy_path} ({size_kb:.0f} KB)")

    gt_path = os.path.join(OUTPUT_DIR, f"{ex['name']}_gt.npy")
    np.save(gt_path, gt_mask)

    manifest.append({
        "name": ex["name"],
        "title": ex["title"],
        "date": ex["date"],
        "hour": ex["hour"],
        "file": f"{ex['name']}.npy",
        "gt_file": f"{ex['name']}_gt.npy",
        "shape": list(crop.shape),
        "convective_pct": round(conv_pct, 1)
    })

# Манифест
with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=lambda x: float(x))

print(f"\nГотово! Примеров: {len(manifest)}")

### 2.3 Скачать примеры (zip)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("examples", "zip", ".", "examples")
files.download("examples.zip")
print("Распакуй examples.zip в корень репозитория goes-storm-watch/")

---
## Готово!

Положи скачанные файлы в репозиторий:
```
goes-storm-watch/
├── model/
│   ├── resnet18_goes.onnx
│   └── norm_stats.json
└── examples/
    ├── manifest.json
    ├── great_plains_storm.npy
    ├── great_plains_storm_gt.npy
    └── ...
```

Закоммить, запушь — GitHub Pages заработает.